# 🚀 DataMate — Zalo AI Challenge 2026: Public Test Inference

Notebook này giúp bạn chạy pipeline suy luận trên tập **`public_test`** (1,000 câu hỏi) bằng **GPU Colab** (T4 / L4 / A100) và xuất ra file **`submission.jsonl`** chuẩn để nộp lên cổng Zalo AI Challenge.

> **Lưu ý:** Hãy bật GPU trước khi chạy: vào **Runtime** → **Change runtime type** → chọn **T4 GPU** hoặc **A100 GPU**.

In [ ]:
# 1. Kiểm tra thông tin GPU
!nvidia-smi

In [ ]:
# 2. Cài đặt các thư viện cần thiết
!pip install -q duckdb>=1.1.0 pandas>=2.2.0 openpyxl python-calamine
!pip install -q transformers>=4.48.0 accelerate>=1.0.0 bitsandbytes>=0.43.0
!pip install -q tqdm loguru unidecode regex

### 3. Kết nối Google Drive & Tự Động Cập Nhật Code
Bạn chỉ cần mount Drive và chạy ô cập nhật code bên dưới để tự động lấy toàn bộ mã nguồn mới nhất từ GitHub.

In [ ]:
# 3.1 Mount Google Drive và trỏ vào thư mục làm việc
from google.colab import drive
drive.mount('/content/drive')

# Chuyển vào thư mục chứa dữ liệu public_test trên Drive
%cd /content/drive/MyDrive/ZALO_AI_Challenge

In [13]:
# 🔄 3.2 CẬP NHẬT CODE MỚI NHẤT TỪ GITHUB (NHÁNH DEV)
# Bấm chạy ô này bất kỳ lúc nào để tự động cập nhật run_public_test.py, src/, ...
# Không cần xóa hay upload lại bằng tay nữa!
import os

REPO_URL = "https://github.com/JimmyK300/zalo-vecna.git"
BRANCH = "dev"

print(f"[*] Đang đồng bộ code từ GitHub ({BRANCH})...")
if not os.path.exists(".git"):
    !git init
    !git config core.filemode false
    !git remote add origin {REPO_URL}
else:
    !git remote set-url origin {REPO_URL}

!git fetch origin {BRANCH}
!git checkout -B {BRANCH} origin/{BRANCH}
!git reset --hard origin/{BRANCH}

print("\n✅ Code đã cập nhật thành công! Commit hiện tại:")
!git log -1 --oneline

[*] Đang đồng bộ code từ GitHub (dev)...
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 8 (delta 6), reused 8 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 4.90 KiB | 0 bytes/s, done.
From https://github.com/JimmyK300/zalo-vecna
 * branch            dev        -> FETCH_HEAD
   d174e4a..095dccb  dev        -> origin/dev
branch 'dev' set up to track 'origin/dev'.
Reset branch 'dev'
Your branch is up to date with 'origin/dev'.
HEAD is now at 095dccb Add diagnostic cell to Colab notebook

✅ Code đã cập nhật thành công! Commit hiện tại:
095dccb (HEAD -> dev, origin/dev) Add diagnostic cell to Colab notebook


In [14]:
# 3.3 Kiểm tra các thư mục và file bắt buộc
import os
print("public_test có sẵn:", os.path.exists("public_test"))
print("src/ có sẵn:", os.path.exists("src"))
print("run_public_test.py có sẵn:", os.path.exists("run_public_test.py"))

public_test có sẵn: True
src/ có sẵn: True
run_public_test.py có sẵn: True


### 4. Chạy Thử Nghiệm 5 Câu (Sanity Check)
Chạy thử 5 câu để kiểm tra mô hình Qwen, DuckDB và Code Execution hoạt động chuẩn xác.

In [15]:
# Chạy thử 5 câu
!python run_public_test.py \
    --data_dir public_test \
    --output submission_sample.jsonl \
    --model_id Qwen/Qwen2.5-Coder-7B-Instruct \
    --load_in_4bit \
    --limit 5

# Xem kết quả 5 câu
!cat submission_sample.jsonl

=== Starting DataMate Inference ===
[*] Using data directory: /content/drive/MyDrive/ZALO_AI_Challenge/public_test
[*] Initializing model: Qwen/Qwen2.5-Coder-7B-Instruct on cuda (4-bit: True)
Loading weights: 100% 339/339 [00:04<00:00, 78.86it/s]
[+] Model loaded successfully!
[*] Processing first 5 questions only.
Inferring: 100% 5/5 [00:40<00:00,  8.03s/it]

[+] Inference completed successfully!
[+] Output saved to: submission_sample.jsonl
[+] Logs saved to: result/logs
{"id": "q_008fdebb6ec9", "type": "string", "answer": "Trà đào cam sả (M)"}
{"id": "q_009187cd2960", "type": "abstain"}
{"id": "q_00bf8df3ea0b", "type": "number", "answer": 4750000}
{"id": "q_00c4e24fa228", "type": "abstain"}
{"id": "q_013b4ff35811", "type": "number", "answer": 208691600}


In [16]:
# 🩺 CHẨN ĐOÁN: cập nhật code + chạy 5 câu + in code/lỗi từng câu
%cd /content/drive/MyDrive/ZALO_AI_Challenge
!git fetch origin dev
!git reset --hard origin/dev
!git log -1 --oneline
!rm -rf result/logs
!python run_public_test.py --data_dir public_test --output submission_sample.jsonl --limit 5 --load_in_4bit
!cat submission_sample.jsonl

import glob, json
for f in sorted(glob.glob("result/logs/*.json")):
    d = json.load(open(f, encoding="utf-8"))
    print("\n=====", d["id"], d["question"])
    print("CODE:\n", d.get("generated_code"))
    print("EXEC_ERROR:", d.get("exec_error"))
    print("ERROR:", d.get("error"))

/content/drive/MyDrive/ZALO_AI_Challenge
From https://github.com/JimmyK300/zalo-vecna
 * branch            dev        -> FETCH_HEAD
HEAD is now at 095dccb Add diagnostic cell to Colab notebook
095dccb (HEAD -> dev, origin/dev) Add diagnostic cell to Colab notebook
=== Starting DataMate Inference ===
[*] Using data directory: /content/drive/MyDrive/ZALO_AI_Challenge/public_test
[*] Initializing model: Qwen/Qwen2.5-Coder-7B-Instruct on cuda (4-bit: True)
Loading weights: 100% 339/339 [00:04<00:00, 78.97it/s]
[+] Model loaded successfully!
[*] Processing first 5 questions only.
Inferring: 100% 5/5 [00:40<00:00,  8.03s/it]

[+] Inference completed successfully!
[+] Output saved to: submission_sample.jsonl
[+] Logs saved to: result/logs
{"id": "q_008fdebb6ec9", "type": "string", "answer": "Trà đào cam sả (M)"}
{"id": "q_009187cd2960", "type": "abstain"}
{"id": "q_00bf8df3ea0b", "type": "number", "answer": 4750000}
{"id": "q_00c4e24fa228", "type": "abstain"}
{"id": "q_013b4ff35811", "type": 

In [ ]:
# 🩺 CHẨN ĐOÁN gọn: chỉ in lỗi của các câu abstain (đọc từ log đã chạy, không chạy lại model)
import glob, json
for f in sorted(glob.glob("result/logs/*.json")):
    d = json.load(open(f, encoding="utf-8"))
    if d["final_submission"].get("type") != "abstain":
        continue
    print("\n=====", d["id"], "|", d["question"])
    print("CODE:\n", (d.get("generated_code") or "")[:1200])
    print("EXEC_ERROR:", (d.get("exec_error") or "")[:600])
    print("ERROR:", (d.get("error") or "")[:600])
    print("STDOUT:", (d.get("captured_stdout") or "")[:300])

In [17]:
# Xem chi tiết log suy luận & code Python đã chạy (hữu ích khi debug)
import glob, json, os
log_files = sorted(glob.glob("result/logs/*.json"))
if log_files:
    print(f"Tổng cộng {len(log_files)} logs. Đang hiển thị log của câu: {os.path.basename(log_files[0])}\n")
    with open(log_files[0], "r", encoding="utf-8") as f:
        print(json.dumps(json.load(f), indent=2, ensure_ascii=False))
else:
    print("Chưa có log nào trong result/logs/")

Tổng cộng 5 logs. Đang hiển thị log của câu: q_008fdebb6ec9.json

{
  "id": "q_008fdebb6ec9",
  "question": "Tháng 11 năm 2025, nhóm trà có mặt hàng nào bán được nhiều ly nhất?",
  "business_id": "dev_002",
  "generated_code": "query = \"\"\"\nSELECT ten_san_pham, SUM(so_luong_san_pham) AS tong_so_luong\nFROM banhang_t11_2025_donhang_chitiet\nWHERE ten_san_pham ILIKE '%trà%'\nGROUP BY ten_san_pham\nORDER BY tong_so_luong DESC\nLIMIT 1;\n\"\"\"\nresult = con.execute(query).fetchone()[0]",
  "elapsed_time": 10.494,
  "final_submission": {
    "id": "q_008fdebb6ec9",
    "type": "string",
    "answer": "Trà đào cam sả (M)"
  }
}


### 5. Chạy Toàn Bộ 1,000 Câu Hỏi (Full Public Test)
Lệnh này sẽ duyệt toàn bộ 1,000 câu hỏi và ghi kết quả liên tục vào file `submission.jsonl`.

In [ ]:
!python run_public_test.py \
    --data_dir public_test \
    --output submission.jsonl \
    --model_id Qwen/Qwen2.5-Coder-7B-Instruct \
    --load_in_4bit

### 6. Kiểm Tra Thống Kê & Tải Kết Quả Về Máy
Kiểm tra đủ số lượng 1,000 dòng và phân bố các type trước khi nộp.

In [ ]:
import os, json
from collections import Counter

if os.path.exists("submission.jsonl"):
    lines = []
    with open("submission.jsonl", "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                lines.append(json.loads(line))

    print(f"Tổng số dòng đã sinh: {len(lines)} / 1000")
    types = Counter([x.get("type") for x in lines])
    print("Phân bố các loại đáp án:")
    for t, c in types.most_common():
        pct = (c / len(lines)) * 100
        print(f"  {t}: {c} ({pct:.1f}%)")
else:
    print("Chưa tìm thấy file submission.jsonl")

In [ ]:
# Tải file submission.jsonl về máy tính để nộp lên cổng thi Zalo
from google.colab import files
if os.path.exists("submission.jsonl"):
    files.download("submission.jsonl")